# Documentation Lead Record

- *Role*: Documentation Lead
- *Assigned Part*: Part 2 - Splitting and Model Building
- *Variables Needed*: X, y
- *Variables Handed Over*: X_train, X_test, y_train, y_test, model

## Part 2: Splitting and model building (Documentation lead)

The `split_data` function separates the features and target into a training set and a test set. The test set is kept aside so the model is judged on data it did not see during training.

In [ ]:
def split_data(X, y):
    """Split the data into training and test sets."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

The `build_preprocessor` function finds the numeric and categorical columns from the dataframe itself. Numeric missing values are filled with the median, and the categorical `Status` column is filled with the most frequent value and then one-hot encoded.

In [ ]:
def build_preprocessor(X):
    """Build the preprocessing steps for numeric and categorical columns."""
    numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_transformer = SimpleImputer(strategy="median")
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="if_binary",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        verbose_feature_names_out=False,
    )
    return preprocessor

In [ ]:
preprocessor = build_preprocessor(X)
preprocessor

The `train_model` function puts the preprocessor, a scaler, and a linear regression model into one pipeline. The pipeline is fitted only on the training data and returned for the later parts.

In [ ]:
def train_model(preprocessor, X_train, y_train):
    """Fit and return the linear regression pipeline."""
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("regressor", LinearRegression()),
        ]
    )
    model.fit(X_train, y_train)
    return model

In [ ]:
model = train_model(preprocessor, X_train, y_train)
model

The imputer and the scaler live inside the pipeline on purpose. When the pipeline is fitted, they learn their values from the training data only, and the same learned values are then applied to the test data. This prevents information from the test set leaking into training.

This part hands over `X_train`, `X_test`, `y_train`, `y_test` and `model`.